In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import random
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import SGD
from concurrent.futures import ThreadPoolExecutor

In [6]:
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
print("Environment set. TF restricted to 1 thread per instance to allow concurrency.")

Environment set. TF restricted to 1 thread per instance to allow concurrency.


In [7]:
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()
X_train=X_train.astype(np.float32) / 255.0
X_test=X_test.astype(np.float32) / 255.0
X_train=X_train.reshape(-1, 28, 28, 1)
X_test=X_test.reshape(-1, 28, 28, 1)

In [8]:
def iid_partition(x, y, num_clients):
    idx=np.random.permutation(len(x))
    x_shuffled, y_shuffled=x[idx], y[idx]
    x_parts=np.array_split(x_shuffled, num_clients)
    y_parts=np.array_split(y_shuffled, num_clients)
    partitions=list(zip(x_parts, y_parts))
    return partitions

In [9]:
def non_iid_partition(x, y, num_clients):
    sorted_idx=np.argsort(y)
    X_sorted, Y_sorted=x[sorted_idx], y[sorted_idx]
    num_shards=2*num_clients
    size_shards=len(x)//num_shards
    shards_idx=np.random.permutation(num_shards)
    partitions=[]
    for i in range(num_clients):
        s1, s2=shards_idx[2*i], shards_idx[2*i+1]
        X_client=np.concatenate((X_sorted[s1*size_shards:(s1+1)*size_shards], X_sorted[s2*size_shards:(s2+1)*size_shards]), axis=0)
        Y_client=np.concatenate((Y_sorted[s1*size_shards:(s1+1)*size_shards], Y_sorted[s2*size_shards:(s2+1)*size_shards]), axis=0)
        idx=np.random.permutation(len(X_client))
        partitions.append((X_client[idx], Y_client[idx]))
    return partitions

In [10]:
def cnn_model():
    model=Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax'),
    ])
    model.compile(optimizer=SGD(learning_rate=0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [11]:
def local_training(client_data, global_weights, mu):
    X_local, Y_local=client_data
    num_sample=len(X_local)
    model=cnn_model()
    model.set_weights(global_weights)
    optimizer=SGD(learning_rate=0.01)
    loss_fn=tf.keras.losses.SparseCategoricalCrossentropy()
    global_weights_tensors=[tf.constant(w) for w in global_weights]
    dataset=tf.data.Dataset.from_tensor_slices((X_local, Y_local)).batch(32)
    
    def train_step(x, y):
        with tf.GradientTape() as tape:
            y_pred=model(x, training=True)
            base_loss=loss_fn(y, y_pred)
            
            proximal_term=0.0
            for local_var, global_var in zip(model.trainable_variables, global_weights_tensors):
                proximal_term+=tf.reduce_sum(tf.square(local_var - global_var))
                
            total_loss=base_loss+(mu/2.0)*proximal_term    
        grads=tape.gradient(total_loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

    for epoch in range(2):
        for batch_x, batch_y in dataset:
            train_step(batch_x, batch_y)
            
    return model.get_weights(), num_sample

In [12]:
def aggregate_weights(client_updates, global_weights):
    tot_sample=sum(num_samples for _, num_samples in client_updates)
    new_weights=[np.zeros_like(w) for w in global_weights]
    
    for local_weights, num_samples in client_updates:
        w_factor=num_samples/tot_sample
        for i in range(len(new_weights)):
            new_weights[i]+=local_weights[i]*w_factor
            
    return new_weights

In [13]:
def evaluate_global(global_model, X_test, Y_test):
    loss, accuracy=global_model.evaluate(X_test, Y_test, verbose=1)
    return loss, accuracy

In [14]:
def run_simulation(partitions, X_test, Y_test, num_clients, rounds, participation_rate, mu):
    global_model=cnn_model()
    global_weights=global_model.get_weights()
    max_workers=min(num_clients, 12)
    history=[]
    
    for r in range(1, rounds + 1):
        num_active=max(1, int(num_clients * participation_rate))
        active_clients=random.sample(range(num_clients), num_active)
        client_updates=[]
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures=[executor.submit(local_training, partitions[i], global_weights, mu) for i in active_clients]
            for future in futures:
                client_updates.append(future.result())
                
        # Aggregate and Update
        global_weights=aggregate_weights(client_updates, global_weights)
        global_model.set_weights(global_weights)
        
        # Evaluate
        loss, acc=evaluate_global(global_model, X_test, Y_test)
        history.append({'round': r, 'loss': loss, 'acc': acc})
        print(f"Round {r} | Accuracy: {acc} | Loss: {loss}")
        
    return history

In [ ]:
CLIENTS=10    
iid_parts = iid_partition(X_train, Y_train, CLIENTS)
non_iid_parts = non_iid_partition(X_train, Y_train, CLIENTS)

In [16]:
IID_ROUNDS=20
NON_IID_ROUNDS=50
MU=0.2
PARTICIPATION_RATE =1
print(f"=== RUNNING IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
iid_history1 = run_simulation(iid_parts, X_test, Y_test, CLIENTS, IID_ROUNDS, PARTICIPATION_RATE, MU)
print(f"\n=== RUNNING NON-IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
non_iid_history1 = run_simulation(non_iid_parts, X_test, Y_test, CLIENTS, NON_IID_ROUNDS, PARTICIPATION_RATE, MU)

=== RUNNING IID FEDPROX (10 Clients, 100% Participation) ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8731 - loss: 0.5348
Round 1 | Accuracy: 0.8730999827384949 | Loss: 0.5348038077354431
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9106 - loss: 0.3167
Round 2 | Accuracy: 0.9106000065803528 | Loss: 0.3166976869106293
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9283 - loss: 0.2495
Round 3 | Accuracy: 0.9283000230789185 | Loss: 0.24950075149536133
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9404 - loss: 0.2089
Round 4 | Accuracy: 0.9404000043869019 | Loss: 0.20887580513954163
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9483 - loss: 0.1799
Round 5 | Accuracy: 0.9483000040054321 | Loss: 0.1798899620771408
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9548 - loss: 0.1580
Round 6 | Accuracy: 0.954800009727478 | Loss: 0.1580309271812439
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9597 - loss: 0.1407
Round 7 | Accura

In [12]:
IID_ROUNDS=50
NON_IID_ROUNDS=100
PARTICIPATION_RATE =0.5
print(f"=== RUNNING IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
iid_history2 = run_simulation(iid_parts, X_test, Y_test, CLIENTS, IID_ROUNDS, PARTICIPATION_RATE, MU)
print(f"\n=== RUNNING NON-IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
non_iid_history2 = run_simulation(non_iid_parts, X_test, Y_test, CLIENTS, NON_IID_ROUNDS, PARTICIPATION_RATE, MU)

=== RUNNING IID FEDPROX (10 Clients, 50.0% Participation) ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8502 - loss: 0.6021
Round 1 | Accuracy: 0.8501999974250793 | Loss: 0.6020631790161133
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9007 - loss: 0.3441
Round 2 | Accuracy: 0.9006999731063843 | Loss: 0.34405970573425293
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9221 - loss: 0.2711
Round 3 | Accuracy: 0.9221000075340271 | Loss: 0.2711368501186371
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9355 - loss: 0.2270
Round 4 | Accuracy: 0.9355000257492065 | Loss: 0.22703127562999725
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9393 - loss: 0.2089
Round 5 | Accuracy: 0.939300000667572 | Loss: 0.20888692140579224
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9491 - loss: 0.1756
Round 6 | Accuracy: 0.9491000175476074 | Loss: 0.17556919157505035
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.9562 - loss: 0.1481
Round 7

In [13]:
IID_ROUNDS=50
NON_IID_ROUNDS=100
PARTICIPATION_RATE =0.3
print(f"=== RUNNING IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
iid_history3 = run_simulation(iid_parts, X_test, Y_test, CLIENTS, IID_ROUNDS, PARTICIPATION_RATE, MU)
print(f"\n=== RUNNING NON-IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
non_iid_history3 = run_simulation(non_iid_parts, X_test, Y_test, CLIENTS, NON_IID_ROUNDS, PARTICIPATION_RATE, MU)

=== RUNNING IID FEDPROX (10 Clients, 30.0% Participation) ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8449 - loss: 0.6082
Round 1 | Accuracy: 0.8449000120162964 | Loss: 0.6082218885421753
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9032 - loss: 0.3426
Round 2 | Accuracy: 0.9031999707221985 | Loss: 0.3425852060317993
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9240 - loss: 0.2601
Round 3 | Accuracy: 0.9240000247955322 | Loss: 0.2601391673088074
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9368 - loss: 0.2241
Round 4 | Accuracy: 0.9368000030517578 | Loss: 0.22407400608062744
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9436 - loss: 0.1906
Round 5 | Accuracy: 0.9435999989509583 | Loss: 0.19063648581504822
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9533 - loss: 0.1656
Round 6 | Accuracy: 0.9532999992370605 | Loss: 0.16563540697097778
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9570 - loss: 0.1476
Round 7

In [17]:
IID_ROUNDS=20
NON_IID_ROUNDS=50
PARTICIPATION_RATE =1
MU=0.5
    
print(f"=== RUNNING IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
iid_history4 = run_simulation(iid_parts, X_test, Y_test, CLIENTS, IID_ROUNDS, PARTICIPATION_RATE, MU)
    
print(f"\n=== RUNNING NON-IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
non_iid_history4 = run_simulation(non_iid_parts, X_test, Y_test, CLIENTS, NON_IID_ROUNDS, PARTICIPATION_RATE, MU)

=== RUNNING IID FEDPROX (10 Clients, 100% Participation) ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8301 - loss: 0.8722
Round 1 | Accuracy: 0.8300999999046326 | Loss: 0.8721870183944702
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8905 - loss: 0.4308
Round 2 | Accuracy: 0.890500009059906 | Loss: 0.4307663142681122
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9123 - loss: 0.3209
Round 3 | Accuracy: 0.9122999906539917 | Loss: 0.3208964467048645
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9239 - loss: 0.2679
Round 4 | Accuracy: 0.9239000082015991 | Loss: 0.267947793006897
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9325 - loss: 0.2338
Round 5 | Accuracy: 0.9325000047683716 | Loss: 0.23379072546958923
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9403 - loss: 0.2082
Round 6 | Accuracy: 0.9402999877929688 | Loss: 0.2081637680530548
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9454 - loss: 0.1881
Round 7 | Ac

In [18]:
IID_ROUNDS=50
NON_IID_ROUNDS=100
PARTICIPATION_RATE =0.5
print(f"=== RUNNING IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
iid_history5 = run_simulation(iid_parts, X_test, Y_test, CLIENTS, IID_ROUNDS, PARTICIPATION_RATE, MU)
print(f"\n=== RUNNING NON-IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
non_iid_history5 = run_simulation(non_iid_parts, X_test, Y_test, CLIENTS, NON_IID_ROUNDS, PARTICIPATION_RATE, MU)

=== RUNNING IID FEDPROX (10 Clients, 50.0% Participation) ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8016 - loss: 0.8600
Round 1 | Accuracy: 0.8015999794006348 | Loss: 0.8599965572357178
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8701 - loss: 0.4731
Round 2 | Accuracy: 0.8701000213623047 | Loss: 0.47310101985931396
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9041 - loss: 0.3457
Round 3 | Accuracy: 0.9041000008583069 | Loss: 0.34570854902267456
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9180 - loss: 0.2910
Round 4 | Accuracy: 0.9179999828338623 | Loss: 0.2910410761833191
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9314 - loss: 0.2368
Round 5 | Accuracy: 0.9314000010490417 | Loss: 0.23683804273605347
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9376 - loss: 0.2151
Round 6 | Accuracy: 0.9376000165939331 | Loss: 0.21509231626987457
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9432 - loss: 0.1907
Round 

In [16]:
IID_ROUNDS=50
NON_IID_ROUNDS=100
PARTICIPATION_RATE =0.3
print(f"=== RUNNING IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
iid_history6 = run_simulation(iid_parts, X_test, Y_test, CLIENTS, IID_ROUNDS, PARTICIPATION_RATE, MU)
print(f"\n=== RUNNING NON-IID FEDPROX ({CLIENTS} Clients, {PARTICIPATION_RATE*100}% Participation) ===")
non_iid_history6 = run_simulation(non_iid_parts, X_test, Y_test, CLIENTS, NON_IID_ROUNDS, PARTICIPATION_RATE, MU)

=== RUNNING IID FEDPROX (10 Clients, 30.0% Participation) ===
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8074 - loss: 0.8529
Round 1 | Accuracy: 0.8073999881744385 | Loss: 0.8528860211372375
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8788 - loss: 0.4562
Round 2 | Accuracy: 0.8787999749183655 | Loss: 0.456159383058548
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9023 - loss: 0.3481
Round 3 | Accuracy: 0.9023000001907349 | Loss: 0.34807640314102173
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9108 - loss: 0.3102
Round 4 | Accuracy: 0.9107999801635742 | Loss: 0.3102385401725769
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9255 - loss: 0.2573
Round 5 | Accuracy: 0.9254999756813049 | Loss: 0.2572866678237915
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9306 - loss: 0.2378
Round 6 | Accuracy: 0.9305999875068665 | Loss: 0.2377970963716507
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9375 - loss: 0.2127
Round 7 | Ac